# GINO Model 3 (Omega+3, 4 channels)

Raw point-cloud learning with `GINO` (no input grid projection): particle inputs from `pair_*_gno`, output queried on fluid nodes.

Split: 80/20 train-test from the same datasets (configurable random/chronological).

In [ ]:
# =========================
# Imports and experiment configuration (GINO on raw point clouds)
# =========================
import glob
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

from neuralop.models import GINO
from neuralop.utils import count_model_params

# Central config: edit here only.
CFG = {
    # Random seed for deterministic split/subsampling and model initialization.
    'seed': 42,
    # Switch for quick preliminary runs before full-scale training.
    'fast_sanity_mode': True,


    # Raw pair directories (input particles + output fluid nodes).
    'data_dirs': [
        '/media/dysco/New Volume/Neeraj/neuralop/data/train/pair_1_gno',
        '/media/dysco/New Volume/Neeraj/neuralop/data/train/pair_2_gno',
        '/media/dysco/New Volume/Neeraj/neuralop/data/train/pair_3_gno',
        '/media/dysco/New Volume/Neeraj/neuralop/data/train/pair_4_gno',
        '/media/dysco/New Volume/Neeraj/neuralop/data/train/pair_5_gno',
    ],

    # Per-directory train/test split.
    'split_mode': 'random',
    # Fraction assigned to train set per directory (remainder goes to test set).
    'train_frac': 0.8,

    # Batch size must stay 1 because particle counts vary by frame.
    'batch_size': 1,
    # Total number of training epochs.
    'epochs': 24,

    # Initial learning rate for AdamW.
    'lr': 3e-4,
    # L2-style regularization weight in AdamW.
    'weight_decay': 1e-4,
    # Evaluate on test split every N epochs.
    'eval_every': 2,


    # Stability and scaling controls.
    'grad_clip_norm': 1.0,
    # Clipping range for omega-proxy feature to avoid extreme values.
    'omega_clip': 50.0,

    # Input point subsampling for speed/memory (None disables).
    'max_input_points': 3000,

    # Maximum number of train output query points sampled per frame.
    'max_train_output_points': 1024,

    # Maximum number of test output query points; None keeps full query set.
    'max_test_output_points': None,

    # Optional cap on training files for fast sanity runs.
    'max_train_files': 256,

    # Optional cap on test files for fast sanity runs.
    'max_test_files': 128,


    # Coordinate normalization bounds are estimated from this many frames (0 => all).
    'bounds_scan_files': 40,


    # GINO / latent FNO config.
    'latent_res': 12,

    # Neighborhood radius for input graph operator lifting.
    'in_gno_radius': 0.06,

    # Neighborhood radius for output graph operator decoding.
    'out_gno_radius': 0.06,

    # Kernel transform variant for input-side GNO layer.
    'in_gno_transform_type': 'nonlinear_kernelonly',
    # Kernel transform variant for output-side GNO layer.
    'out_gno_transform_type': 'linear',
    # Feature width in GNO embedding layers.
    'gno_embed_channels': 24,

    # Fourier mode counts used by latent FNO inside GINO.
    'fno_n_modes': (4, 4, 4),

    # Hidden-channel width of latent FNO inside GINO.
    'fno_hidden_channels': 32,

    # Number of latent FNO layers inside GINO.
    'fno_n_layers': 3,

    # Projection expansion ratio used in latent FNO blocks.
    'projection_channel_ratio': 2,


    # Variant-specific fields.
    'input_channels': 4,
    # Feature construction mode: omega-only, 3-ch no-omega, or 4-ch omega+3.
    'input_variant': 'omega4',
    # Prefix used when saving checkpoints/figures for this run.
    'file_tag': 'gino_model3_3d',
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if CFG['batch_size'] != 1:
    print('[warn] GINO raw-pointcloud pipeline requires batch_size=1 with variable geometry; forcing batch_size=1.')
    CFG['batch_size'] = 1

print('Device:', DEVICE)
print('Variant:', CFG['input_variant'], 'channels=', CFG['input_channels'])
print('Split mode:', CFG['split_mode'])
print('Fast sanity mode:', CFG['fast_sanity_mode'])
print('Train/Test file caps:', CFG['max_train_files'], CFG['max_test_files'])
print('Train/Test output-query caps:', CFG['max_train_output_points'], CFG['max_test_output_points'])
if DEVICE == 'cpu':
    print('[warn] Training on CPU can be very slow for GINO; prefer CUDA or reduce max_input_points/epochs.')


In [ ]:
# =========================
# Data setup (raw pair_*_gno) with 80/20 train-test split
# =========================
cwd = Path.cwd().resolve()
if (cwd / '.git').exists():
    REPO_ROOT = cwd
elif (cwd.parent / '.git').exists():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

print('Detected REPO_ROOT:', REPO_ROOT)

RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results directory:', RESULTS_DIR)


def save_fig(fig, filename, dpi=220):
    out = RESULTS_DIR / f"{CFG['file_tag']}_{filename}"
    fig.savefig(out, dpi=dpi, bbox_inches='tight')
    print(f'[saved] {out}')


def resolve_dirs(paths, must_exist=True):
    dirs = []
    for d in paths:
        p = (REPO_ROOT / d).resolve()
        if p.is_dir():
            dirs.append(p)
        elif must_exist:
            raise FileNotFoundError(f'Missing directory: {p}')
        else:
            print(f'[warn] missing directory, skipping: {p}')
    return dirs


DATA_DIRS = resolve_dirs(CFG['data_dirs'], must_exist=True)


def frame_num_from_name(path_str):
    parts = Path(path_str).stem.split('_')
    for tok in parts:
        if tok.isdigit():
            return int(tok)
    return 10**12


def list_raw_frame_files(sim_dir):
    files = sorted(glob.glob(str(sim_dir / 'frame_*.npz')), key=frame_num_from_name)
    # Safety: exclude already-gridded files if present in same directory.
    files = [f for f in files if not Path(f).stem.endswith('_grid')]
    return files


def raw_file_is_valid(file_path, min_particles=2):
    required = ['pos', 'nodes', 'U_nodes', 'gamma_mag', 'sigma', 'vol']
    try:
        d = np.load(file_path)
    except Exception as exc:
        return False, f'load_error: {exc}'

    for k in required:
        if k not in d:
            return False, f'missing_key:{k}'

    pos = np.asarray(d['pos'])
    nodes = np.asarray(d['nodes'])
    y = np.asarray(d['U_nodes'])
    gamma = np.asarray(d['gamma_mag']).reshape(-1)
    sigma = np.asarray(d['sigma']).reshape(-1)
    vol = np.asarray(d['vol']).reshape(-1)

    if pos.ndim != 2 or pos.shape[1] < 3:
        return False, f'bad_pos_shape:{tuple(pos.shape)}'
    if nodes.ndim != 2 or nodes.shape[1] < 3:
        return False, f'bad_nodes_shape:{tuple(nodes.shape)}'
    if y.ndim != 2 or y.shape[1] < 3:
        return False, f'bad_target_shape:{tuple(y.shape)}'

    n = pos.shape[0]
    if n < min_particles:
        return False, f'too_few_particles:{n}'
    if gamma.shape[0] != n or sigma.shape[0] != n or vol.shape[0] != n:
        return False, f'feature_length_mismatch:n={n},gamma={gamma.shape[0]},sigma={sigma.shape[0]},vol={vol.shape[0]}'

    if not np.isfinite(pos[:, :3]).all():
        return False, 'non_finite_pos'
    if not np.isfinite(nodes[:, :3]).all():
        return False, 'non_finite_nodes'
    if not np.isfinite(y[:, :3]).all():
        return False, 'non_finite_target'
    if not np.isfinite(gamma).all():
        return False, 'non_finite_gamma'
    if not np.isfinite(sigma).all():
        return False, 'non_finite_sigma'
    if not np.isfinite(vol).all():
        return False, 'non_finite_vol'

    return True, 'ok'


def filter_valid_raw_files(files, label, min_particles=2, max_report=5):
    valid = []
    dropped = []
    for fp in files:
        ok, reason = raw_file_is_valid(fp, min_particles=min_particles)
        if ok:
            valid.append(fp)
        else:
            dropped.append((fp, reason))

    if dropped:
        print(f"[warn] {label}: dropped {len(dropped)} invalid files")
        for fp, reason in dropped[:max_report]:
            print(f"    - {Path(fp).name}: {reason}")
        if len(dropped) > max_report:
            print(f"    ... and {len(dropped) - max_report} more")

    return valid


def split_train_test(files, train_frac=0.8, mode='random', seed=42):
    files = list(files)
    n = len(files)
    if n == 0:
        return [], []
    if n == 1:
        return files, []

    if mode == 'random':
        rng = np.random.default_rng(seed)
        idx = np.arange(n)
        rng.shuffle(idx)
        files = [files[i] for i in idx]
    elif mode == 'chronological':
        pass
    else:
        raise ValueError("split_mode must be 'random' or 'chronological'")

    n_train = max(1, int(train_frac * n))
    n_train = min(n_train, n - 1)
    return files[:n_train], files[n_train:]


TRAIN_FILES, TEST_FILES = [], []
for i, d in enumerate(DATA_DIRS):
    files = list_raw_frame_files(d)
    if len(files) == 0:
        print(f'[warn] no frame_*.npz found in {d}')
        continue

    files = filter_valid_raw_files(files, label=f'{d.name}', min_particles=2)
    if len(files) == 0:
        print(f'[warn] no valid raw files remain in {d}')
        continue

    tr, te = split_train_test(
        files,
        train_frac=CFG['train_frac'],
        mode=CFG['split_mode'],
        seed=CFG['seed'] + i,
    )
    TRAIN_FILES.extend(tr)
    TEST_FILES.extend(te)
    print(f'[info] {d.name}: valid_total={len(files)}, train={len(tr)}, test={len(te)}')

if len(TRAIN_FILES) == 0:
    raise RuntimeError('No training files found after filtering. Check CFG[data_dirs].')
if len(TEST_FILES) == 0:
    raise RuntimeError('No test files found after split/filtering. Increase data or adjust train_frac.')

if set(TRAIN_FILES).intersection(set(TEST_FILES)):
    raise RuntimeError('Train/test overlap detected.')

if CFG.get('max_train_files') and len(TRAIN_FILES) > CFG['max_train_files']:
    rng = np.random.default_rng(CFG['seed'] + 1701)
    idx = rng.choice(len(TRAIN_FILES), size=CFG['max_train_files'], replace=False)
    TRAIN_FILES = [TRAIN_FILES[i] for i in sorted(idx.tolist())]
    print(f"[info] train file cap active: using {len(TRAIN_FILES)} files")

if CFG.get('max_test_files') and len(TEST_FILES) > CFG['max_test_files']:
    rng = np.random.default_rng(CFG['seed'] + 2701)
    idx = rng.choice(len(TEST_FILES), size=CFG['max_test_files'], replace=False)
    TEST_FILES = [TEST_FILES[i] for i in sorted(idx.tolist())]
    print(f"[info] test file cap active: using {len(TEST_FILES)} files")

print(f'Train files: {len(TRAIN_FILES)} | Test files: {len(TEST_FILES)}')


def scan_coord_bounds(files, max_files=80):
    if max_files and max_files > 0 and len(files) > max_files:
        idx = np.linspace(0, len(files) - 1, max_files, dtype=int)
        scan_files = [files[i] for i in idx]
    else:
        scan_files = list(files)

    cmin = np.array([np.inf, np.inf, np.inf], dtype=np.float64)
    cmax = np.array([-np.inf, -np.inf, -np.inf], dtype=np.float64)

    for f in scan_files:
        d = np.load(f)
        pos = np.asarray(d['pos'][:, :3], dtype=np.float64)
        nodes = np.asarray(d['nodes'][:, :3], dtype=np.float64)

        pmin = np.minimum(pos.min(axis=0), nodes.min(axis=0))
        pmax = np.maximum(pos.max(axis=0), nodes.max(axis=0))
        cmin = np.minimum(cmin, pmin)
        cmax = np.maximum(cmax, pmax)

    return cmin.astype(np.float32), cmax.astype(np.float32), len(scan_files)


COORD_MIN, COORD_MAX, SCANNED = scan_coord_bounds(
    TRAIN_FILES + TEST_FILES,
    max_files=CFG['bounds_scan_files'],
)
COORD_SPAN = np.maximum(COORD_MAX - COORD_MIN, 1e-12)
print('Coord bounds from', SCANNED, 'files')
print('  min:', COORD_MIN)
print('  max:', COORD_MAX)


def normalize_points(xyz):
    xyz_n = (xyz - COORD_MIN) / COORD_SPAN
    return np.clip(xyz_n, 0.0, 1.0)


def omega_proxy_from_particles(gamma_mag, sigma, sigma_floor=1e-6):
    # Robust proxy from Gaussian core center value: omega ~= gamma / (pi^(3/2) * sigma^3).
    # Use |sigma| and log-compression to control extreme dynamic range.
    s = np.clip(np.abs(sigma), sigma_floor, None)
    omega = gamma_mag / ((np.pi ** 1.5) * (s ** 3))
    omega = np.sign(omega) * np.log1p(np.abs(omega))
    omega = np.clip(omega, -CFG['omega_clip'], CFG['omega_clip'])
    return omega


def build_input_features(d, variant):
    gamma = np.asarray(d['gamma_mag'], dtype=np.float32).reshape(-1)
    sigma = np.asarray(d['sigma'], dtype=np.float32).reshape(-1)
    vol = np.asarray(d['vol'], dtype=np.float32).reshape(-1)
    omega = omega_proxy_from_particles(gamma, sigma)

    if variant == 'omega':
        feats = omega[:, None]
    elif variant == 'no_omega3':
        feats = np.stack([gamma, sigma, vol], axis=1)
    elif variant == 'omega4':
        feats = np.stack([omega, gamma, sigma, vol], axis=1)
    else:
        raise ValueError("input_variant must be one of: 'omega', 'no_omega3', 'omega4'")

    return feats.astype(np.float32)




def estimate_feature_stats(train_files, variant, max_files=80):
    if max_files and max_files > 0 and len(train_files) > max_files:
        idx = np.linspace(0, len(train_files) - 1, max_files, dtype=int)
        files = [train_files[i] for i in idx]
    else:
        files = list(train_files)

    s1 = None
    s2 = None
    n = 0

    for fp in files:
        d = np.load(fp)
        x = build_input_features(d, variant).astype(np.float64)
        if s1 is None:
            s1 = x.sum(axis=0)
            s2 = (x ** 2).sum(axis=0)
        else:
            s1 += x.sum(axis=0)
            s2 += (x ** 2).sum(axis=0)
        n += x.shape[0]

    mean = (s1 / max(n, 1)).astype(np.float32)
    var = (s2 / max(n, 1) - mean.astype(np.float64) ** 2).astype(np.float32)
    std = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)
    return mean, std


FEAT_MEAN, FEAT_STD = estimate_feature_stats(TRAIN_FILES, CFG['input_variant'], max_files=80)
print('Feature mean:', FEAT_MEAN)
print('Feature std :', FEAT_STD)

def load_sample(file_path, max_input_points=None, max_output_points=None, split_tag='train'):
    d = np.load(file_path)

    pos = np.asarray(d['pos'][:, :3], dtype=np.float32)
    nodes = np.asarray(d['nodes'][:, :3], dtype=np.float32)
    y = np.asarray(d['U_nodes'][:, :3], dtype=np.float32)
    x = build_input_features(d, CFG['input_variant'])
    x = (x - FEAT_MEAN[None, :]) / FEAT_STD[None, :]
    x = np.clip(x, -8.0, 8.0).astype(np.float32)

    if x.shape[0] != pos.shape[0]:
        raise RuntimeError(f'Feature/position mismatch in {file_path}: x={x.shape}, pos={pos.shape}')

    frame = int(np.asarray(d['frame']).item()) if 'frame' in d else frame_num_from_name(file_path)

    if max_input_points is not None and max_input_points > 0 and pos.shape[0] > max_input_points:
        rng = np.random.default_rng(CFG['seed'] + frame)
        idx = rng.choice(pos.shape[0], size=max_input_points, replace=False)
        pos = pos[idx]
        x = x[idx]

    pos_n = normalize_points(pos).astype(np.float32)
    nodes_n = normalize_points(nodes).astype(np.float32)

    if max_output_points is not None and max_output_points > 0 and nodes_n.shape[0] > max_output_points:
        split_offset = 100000 if split_tag == 'test' else 0
        rng = np.random.default_rng(CFG['seed'] + split_offset + frame)
        idx = rng.choice(nodes_n.shape[0], size=max_output_points, replace=False)
        nodes_n = nodes_n[idx]
        y = y[idx]

    return {
        'input_geom': torch.tensor(pos_n, dtype=torch.float32),
        'x': torch.tensor(x, dtype=torch.float32),
        'output_queries': torch.tensor(nodes_n, dtype=torch.float32),
        'y': torch.tensor(y, dtype=torch.float32),
        'frame': torch.tensor(frame, dtype=torch.long),
        'file': str(file_path),
    }


class RawPairDataset(Dataset):
    def __init__(self, files, max_input_points=None, max_output_points=None, split_tag='train'):
        self.files = list(files)
        self.max_input_points = max_input_points
        self.max_output_points = max_output_points
        self.split_tag = split_tag

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        return load_sample(
            self.files[idx],
            max_input_points=self.max_input_points,
            max_output_points=self.max_output_points,
            split_tag=self.split_tag,
        )


TRAIN_DS = RawPairDataset(
    TRAIN_FILES,
    max_input_points=CFG['max_input_points'],
    max_output_points=CFG['max_train_output_points'],
    split_tag='train',
)
TEST_DS = RawPairDataset(
    TEST_FILES,
    max_input_points=CFG['max_input_points'],
    max_output_points=CFG['max_test_output_points'],
    split_tag='test',
)

TRAIN_LOADER = DataLoader(TRAIN_DS, batch_size=CFG['batch_size'], shuffle=True)
TEST_LOADER = DataLoader(TEST_DS, batch_size=CFG['batch_size'], shuffle=False)

s_tr = TRAIN_DS[0]
s_te = TEST_DS[0]
print('Train sample -> input_geom:', tuple(s_tr['input_geom'].shape), 'x:', tuple(s_tr['x'].shape), 'y:', tuple(s_tr['y'].shape))
print('Test sample  -> input_geom:', tuple(s_te['input_geom'].shape), 'x:', tuple(s_te['x'].shape), 'y:', tuple(s_te['y'].shape))


In [ ]:
# =========================
# GINO model + optimizer + metrics
# =========================
def make_latent_queries(res, device):
    lin = torch.linspace(0.0, 1.0, res, device=device, dtype=torch.float32)
    X, Y, Z = torch.meshgrid(lin, lin, lin, indexing='ij')
    return torch.stack([X, Y, Z], dim=-1).unsqueeze(0)  # [1, nx, ny, nz, 3]


LATENT_QUERIES = make_latent_queries(CFG['latent_res'], DEVICE)
print('LATENT_QUERIES shape:', tuple(LATENT_QUERIES.shape))


def build_model(cfg):
    return GINO(
        in_channels=cfg['input_channels'],
        out_channels=3,
        gno_coord_dim=3,
        in_gno_radius=cfg['in_gno_radius'],
        out_gno_radius=cfg['out_gno_radius'],
        in_gno_transform_type=cfg['in_gno_transform_type'],
        out_gno_transform_type=cfg['out_gno_transform_type'],
        gno_embed_channels=cfg['gno_embed_channels'],
        gno_use_open3d=False,
        gno_use_torch_scatter=False,
        fno_n_modes=cfg['fno_n_modes'],
        fno_hidden_channels=cfg['fno_hidden_channels'],
        fno_n_layers=cfg['fno_n_layers'],
        projection_channel_ratio=cfg['projection_channel_ratio'],
    ).to(DEVICE)


MODEL = build_model(CFG)

TOTAL_PARAMS = count_model_params(MODEL)
TRAINABLE_PARAMS = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print('Total params:', TOTAL_PARAMS)
print('Trainable params:', TRAINABLE_PARAMS)

OPT = torch.optim.AdamW(MODEL.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
SCH = torch.optim.lr_scheduler.CosineAnnealingLR(OPT, T_max=CFG['epochs'])
LOSS_FN = torch.nn.MSELoss()


def predict(model, batch):
    return model(
        input_geom=batch['input_geom'].to(DEVICE),
        latent_queries=LATENT_QUERIES,
        output_queries=batch['output_queries'].to(DEVICE),
        x=batch['x'].to(DEVICE),
    )


def batch_metrics(pred, target):
    diff = pred - target
    mse = torch.mean(diff ** 2, dim=(1, 2))
    mae = torch.mean(torch.abs(diff), dim=(1, 2))
    rmse = torch.sqrt(mse)

    diff_flat = diff.reshape(diff.shape[0], -1)
    tgt_flat = target.reshape(target.shape[0], -1)
    rel_l2 = torch.linalg.norm(diff_flat, dim=1) / torch.linalg.norm(tgt_flat, dim=1).clamp_min(1e-12)

    return {
        'mse': mse.mean().item(),
        'mae': mae.mean().item(),
        'rmse': rmse.mean().item(),
        'rel_l2': rel_l2.mean().item(),
        'rel_vec': rel_l2.detach().cpu().numpy(),
    }


@torch.no_grad()
def evaluate(model, loader, collect=False):
    model.eval()
    acc = {'mse': 0.0, 'mae': 0.0, 'rmse': 0.0, 'rel_l2': 0.0}
    rels = []
    n = 0

    for batch in loader:
        pred = predict(model, batch)
        y = batch['y'].to(DEVICE)
        m = batch_metrics(pred, y)
        for k in acc:
            acc[k] += m[k]
        if collect:
            rels.extend(m['rel_vec'].tolist())
        n += 1

    for k in acc:
        acc[k] /= max(1, n)
    if collect:
        acc['rel_per_sample'] = np.asarray(rels, dtype=np.float32)
    return acc


In [ ]:
# =========================
# Training loop (80/20 train-test)
# =========================
HIST = {
    # Epoch index used on convergence plot x-axis.
    'epoch': [],
    # Training MSE averaged over one epoch.
    'train_mse': [],
    # Training relative L2 error averaged over one epoch.
    'train_rel_l2': [],
    # Test MSE at each evaluation epoch.
    'test_mse': [],
    # Test relative L2 at each evaluation epoch.
    'test_rel_l2': [],
    # Test RMSE at each evaluation epoch.
    'test_rmse': [],
    # Learning-rate history recorded from scheduler.
    'lr': [],
    # Count of batches skipped due to non-finite loss each epoch.
    'bad_batches': [],
}

BEST_TEST = float('inf')
CKPT_PATH = RESULTS_DIR / f"{CFG['file_tag']}_best_model.pt"

for epoch in range(1, CFG['epochs'] + 1):
    MODEL.train()
    train_loss = 0.0
    train_rel = 0.0
    n = 0
    bad_batches = 0

    for batch in TRAIN_LOADER:
        OPT.zero_grad(set_to_none=True)
        pred = predict(MODEL, batch)
        y = batch['y'].to(DEVICE)

        loss = LOSS_FN(pred, y)
        if not torch.isfinite(loss):
            bad_batches += 1
            continue

        loss.backward()

        clip_norm = CFG.get('grad_clip_norm', None)
        if clip_norm is not None and clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(MODEL.parameters(), max_norm=clip_norm)

        OPT.step()

        with torch.no_grad():
            rel = batch_metrics(pred, y)['rel_l2']

        train_loss += loss.item()
        train_rel += rel
        n += 1

    SCH.step()

    if n == 0:
        raise RuntimeError(
            f'All batches produced non-finite loss in epoch {epoch}. '
            'Reduce lr, reduce radii, lower max_input_points, or inspect feature scaling.'
        )

    if epoch % CFG['eval_every'] == 0:
        train_mse = train_loss / n
        train_rel_l2 = train_rel / n
        test_m = evaluate(MODEL, TEST_LOADER)

        HIST['epoch'].append(epoch)
        HIST['train_mse'].append(train_mse)
        HIST['train_rel_l2'].append(train_rel_l2)
        HIST['test_mse'].append(test_m['mse'])
        HIST['test_rel_l2'].append(test_m['rel_l2'])
        HIST['test_rmse'].append(test_m['rmse'])
        HIST['lr'].append(OPT.param_groups[0]['lr'])
        HIST['bad_batches'].append(bad_batches)

        if np.isfinite(test_m['rel_l2']) and test_m['rel_l2'] < BEST_TEST:
            BEST_TEST = test_m['rel_l2']
            torch.save(
                {
                    'model_state_dict': MODEL.state_dict(),
                    'config': dict(CFG),
                    'best_test_rel_l2': BEST_TEST,
                },
                CKPT_PATH,
            )

        print(
            f"[{epoch:03d}] train_mse={train_mse:.4e} train_rel_l2={train_rel_l2:.4e} "
            f"test_rel_l2={test_m['rel_l2']:.4e} test_rmse={test_m['rmse']:.4e} "
            f"bad_batches={bad_batches} lr={HIST['lr'][-1]:.2e}"
        )

print(f"Best test rel-L2: {BEST_TEST:.4e}")
print(f"Checkpoint: {CKPT_PATH}")


In [ ]:
# =========================
# Post-training test snippet (load best checkpoint and evaluate)
# =========================
# PyTorch >=2.6 defaults to weights_only=True. For our own trusted checkpoint,
# explicitly allow full unpickling so optimizer/scheduler metadata can be read.
try:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    # Backward compatibility with older torch versions lacking weights_only arg.
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
MODEL.load_state_dict(state_dict)
MODEL.eval()

TEST_METRICS = evaluate(MODEL, TEST_LOADER, collect=True)
print('Loaded checkpoint:', CKPT_PATH)
if isinstance(ckpt, dict):
    print('Checkpoint best test rel-L2:', ckpt.get('best_test_rel_l2', 'n/a'))
else:
    print('Checkpoint best test rel-L2: n/a (state_dict only)')
print(
    f"Post-train TEST metrics -> rel-L2={TEST_METRICS['rel_l2']:.4e}, "
    f"RMSE={TEST_METRICS['rmse']:.4e}, MAE={TEST_METRICS['mae']:.4e}, MSE={TEST_METRICS['mse']:.4e}"
)


In [ ]:
# =========================
# Core plots: params + convergence
# =========================
module_param_counts = defaultdict(int)
for name, p in MODEL.named_parameters():
    module_param_counts[name.split('.')[0]] += p.numel()

mods = list(module_param_counts.keys())
vals = [module_param_counts[m] for m in mods]
order = np.argsort(vals)[::-1]
mods = [mods[i] for i in order]
vals = [vals[i] for i in order]

fig_p, ax_p = plt.subplots(figsize=(10, 4))
ax_p.bar(mods, vals)
ax_p.set_title(f"Parameter distribution (total={TOTAL_PARAMS:,})")
ax_p.set_ylabel('Parameters')
ax_p.tick_params(axis='x', rotation=45)
fig_p.tight_layout()
save_fig(fig_p, 'params_by_module.png')
plt.show()

fig_c, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].plot(HIST['epoch'], HIST['train_mse'], label='train MSE')
axs[0].plot(HIST['epoch'], HIST['test_mse'], label='test MSE')
axs[0].set_yscale('log')
axs[0].set_xlabel('Epoch')
axs[0].set_title('MSE convergence')
axs[0].legend()

axs[1].plot(HIST['epoch'], HIST['train_rel_l2'], label='train rel-L2')
axs[1].plot(HIST['epoch'], HIST['test_rel_l2'], label='test rel-L2')
axs[1].set_yscale('log')
axs[1].set_xlabel('Epoch')
axs[1].set_title('Relative error convergence')
axs[1].legend()

axs[2].plot(HIST['epoch'], HIST['lr'], label='LR')
axs[2].set_xlabel('Epoch')
axs[2].set_title('Learning-rate schedule')
axs[2].legend()

fig_c.tight_layout()
save_fig(fig_c, 'convergence_curves.png')
plt.show()


In [ ]:
# =========================
# Error-distribution + parity diagnostics
# =========================
if 'TEST_METRICS' not in globals() or 'rel_per_sample' not in TEST_METRICS:
    TEST_METRICS = evaluate(MODEL, TEST_LOADER, collect=True)

rel_vals = np.asarray(TEST_METRICS['rel_per_sample'], dtype=np.float32)

sample = next(iter(TEST_LOADER))
with torch.no_grad():
    pred = predict(MODEL, sample)[0].detach().cpu()

true = sample['y'][0].detach().cpu()
true_speed = torch.linalg.norm(true, dim=-1).numpy()
pred_speed = torch.linalg.norm(pred, dim=-1).numpy()

rng = np.random.default_rng(CFG['seed'])
if true_speed.size > 12000:
    idx = rng.choice(true_speed.size, size=12000, replace=False)
    true_speed = true_speed[idx]
    pred_speed = pred_speed[idx]

fig_d, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].hist(rel_vals, bins=30, color='tab:blue', alpha=0.8)
axs[0].set_title('TEST rel-L2 histogram')
axs[0].set_xlabel('rel-L2')
axs[0].set_ylabel('count')

sorted_rel = np.sort(rel_vals)
cdf = np.linspace(0, 1, len(sorted_rel), endpoint=True)
axs[1].plot(sorted_rel, cdf, color='tab:green')
axs[1].set_title('TEST rel-L2 CDF')
axs[1].set_xlabel('rel-L2')
axs[1].set_ylabel('cdf')
axs[1].grid(True, alpha=0.3)

vmin = min(float(true_speed.min()), float(pred_speed.min()))
vmax = max(float(true_speed.max()), float(pred_speed.max()))
axs[2].scatter(true_speed, pred_speed, s=6, alpha=0.35)
axs[2].plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=1.4)
axs[2].set_title('Parity plot (speed magnitude)')
axs[2].set_xlabel('true speed')
axs[2].set_ylabel('pred speed')

fig_d.tight_layout()
save_fig(fig_d, 'error_distribution_and_parity.png')
plt.show()


In [ ]:
# =========================
# Chronological snapshot panel (mid-plane scatter)
# =========================
def extract_midplane(points_xyz, values, band_frac=0.04):
    z = points_xyz[:, 2]
    zmin, zmax = float(z.min()), float(z.max())
    zmid = 0.5 * (zmin + zmax)
    band = max((zmax - zmin) * band_frac, 1e-6)
    mask = np.abs(z - zmid) <= band
    if mask.sum() == 0:
        # fallback: take closest 5%
        idx = np.argsort(np.abs(z - zmid))[: max(1, int(0.05 * len(z)))]
        mask = np.zeros_like(z, dtype=bool)
        mask[idx] = True
    return points_xyz[mask], values[mask]


def load_for_plot(file_path):
    s = load_sample(
        file_path,
        max_input_points=CFG['max_input_points'],
        max_output_points=CFG['max_test_output_points'],
        split_tag='test',
    )
    batch = {
        'input_geom': s['input_geom'].unsqueeze(0),
        'x': s['x'].unsqueeze(0),
        'output_queries': s['output_queries'].unsqueeze(0),
        'y': s['y'].unsqueeze(0),
    }
    with torch.no_grad():
        pred = predict(MODEL, batch)[0].detach().cpu().numpy()

    y = s['y'].cpu().numpy()
    pts = s['output_queries'].cpu().numpy()

    ts = np.linalg.norm(y, axis=1)
    ps = np.linalg.norm(pred, axis=1)
    es = np.abs(ps - ts)

    rel = np.linalg.norm((pred - y).reshape(-1)) / max(np.linalg.norm(y.reshape(-1)), 1e-12)
    return pts, ts, ps, es, rel, int(s['frame'].item())


def plot_snapshot_panel(files, n_times=4, panel_name='TEST'):
    files = sorted(files, key=frame_num_from_name)
    n_times = max(1, min(n_times, len(files)))
    idxs = np.linspace(0, len(files) - 1, n_times, dtype=int)
    chosen = [files[i] for i in idxs]

    fig, axs = plt.subplots(3, n_times, figsize=(4 * n_times, 9))
    if n_times == 1:
        axs = np.asarray(axs).reshape(3, 1)

    for j, f in enumerate(chosen):
        pts, ts, ps, es, rel, fid = load_for_plot(f)

        p2, t2 = extract_midplane(pts, ts)
        _, p2v = extract_midplane(pts, ps)
        _, e2v = extract_midplane(pts, es)

        im0 = axs[0, j].scatter(p2[:, 0], p2[:, 1], c=t2, s=8, cmap='turbo')
        axs[0, j].set_title(f't={fid} true')
        axs[0, j].set_aspect('equal')
        plt.colorbar(im0, ax=axs[0, j], fraction=0.046, pad=0.04)

        im1 = axs[1, j].scatter(p2[:, 0], p2[:, 1], c=p2v, s=8, cmap='turbo')
        axs[1, j].set_title('pred')
        axs[1, j].set_aspect('equal')
        plt.colorbar(im1, ax=axs[1, j], fraction=0.046, pad=0.04)

        im2 = axs[2, j].scatter(p2[:, 0], p2[:, 1], c=e2v, s=8, cmap='magma')
        axs[2, j].set_title(f'|error| rel-L2={rel:.2e}')
        axs[2, j].set_aspect('equal')
        plt.colorbar(im2, ax=axs[2, j], fraction=0.046, pad=0.04)

    fig.suptitle(f'{panel_name}: mid-plane snapshots', fontsize=14, y=0.995)
    fig.tight_layout()
    save_fig(fig, f'snapshots_{panel_name.lower()}.png')
    plt.show()


plot_snapshot_panel(TEST_FILES, n_times=4, panel_name='TEST')


In [ ]:
# =========================
# Data-hunger curve (subset size vs test error)
# =========================
def train_eval_subset(train_files_subset, cfg, quick_epochs=3):
    ds = RawPairDataset(
        train_files_subset,
        max_input_points=cfg['max_input_points'],
        max_output_points=cfg.get('max_train_output_points', None),
        split_tag='train',
    )
    ld = DataLoader(ds, batch_size=1, shuffle=True)

    m = build_model(cfg)
    opt = torch.optim.AdamW(m.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=quick_epochs)
    loss_fn = torch.nn.MSELoss()

    for _ in range(quick_epochs):
        m.train()
        for batch in ld:
            opt.zero_grad(set_to_none=True)
            pred = m(
                input_geom=batch['input_geom'].to(DEVICE),
                latent_queries=LATENT_QUERIES,
                output_queries=batch['output_queries'].to(DEVICE),
                x=batch['x'].to(DEVICE),
            )
            y = batch['y'].to(DEVICE)
            loss = loss_fn(pred, y)
            loss.backward()
            opt.step()
        sch.step()

    tm = evaluate(m, TEST_LOADER)
    return tm['rel_l2']


def run_data_hunger_curve(fractions=(0.2, 0.5, 1.0), repeats=1, quick_epochs=3):
    rng = np.random.default_rng(CFG['seed'])
    n_total = len(TRAIN_FILES)
    curve = []

    for frac in fractions:
        n_use = max(1, int(frac * n_total))
        vals = []
        for _ in range(repeats):
            idx = rng.choice(n_total, size=n_use, replace=False)
            subset = [TRAIN_FILES[i] for i in sorted(idx.tolist())]
            rel = train_eval_subset(subset, dict(CFG), quick_epochs=quick_epochs)
            vals.append(rel)
        curve.append((frac, n_use, float(np.mean(vals)), float(np.std(vals))))
        print(f"frac={frac:.2f}, n={n_use}, test rel-L2 mean={np.mean(vals):.4e}, std={np.std(vals):.2e}")

    ns = [c[1] for c in curve]
    means = [c[2] for c in curve]
    stds = [c[3] for c in curve]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(ns, means, yerr=stds, marker='o', capsize=4)
    ax.set_xlabel('Number of training samples')
    ax.set_ylabel('Test rel-L2')
    ax.set_title('Data-hunger curve')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_fig(fig, 'data_hunger_curve.png')
    plt.show()

    return curve


DATA_HUNGER = run_data_hunger_curve(fractions=(0.2, 0.5, 1.0), repeats=1, quick_epochs=3)


In [ ]:
# =========================
# Quick convergence sensitivity sweeps
# =========================
def cfg_with(base_cfg, key, value):
    c = dict(base_cfg)
    c[key] = value
    return c


def quick_sweep(param_name, values, quick_epochs=6):
    results = []
    for v in values:
        c = cfg_with(CFG, param_name, v)
        rel = train_eval_subset(TRAIN_FILES, c, quick_epochs=quick_epochs)
        results.append((v, rel))
        print(f"{param_name}={v} -> test rel-L2 {rel:.4e}")

    xs = [r[0] for r in results]
    ys = [r[1] for r in results]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xs, ys, marker='o')
    ax.set_xlabel(param_name)
    ax.set_ylabel('Final test rel-L2')
    ax.set_title(f'Quick sweep: {param_name}')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_fig(fig, f'sweep_{param_name}.png')
    plt.show()
    return results


SWEEP_HIDDEN = quick_sweep('fno_hidden_channels', [32, 48, 64, 96], quick_epochs=6)
SWEEP_RADIUS = quick_sweep('in_gno_radius', [0.04, 0.06, 0.08, 0.12], quick_epochs=6)


In [ ]:
# =========================
# 3D Data Profile: pairs, channels, datapoints
# =========================
def gino_input_channel_labels(cfg):
    variant = cfg.get('input_variant', '')
    mapping = {
        'omega': ['omega_proxy'],
        'no_omega3': ['gamma_mag', 'sigma', 'vol'],
        'omega4': ['omega_proxy', 'gamma_mag', 'sigma', 'vol'],
    }
    labels = mapping.get(variant, [f'ch_{i}' for i in range(cfg['input_channels'])])
    if len(labels) != cfg['input_channels']:
        labels = [f'ch_{i}' for i in range(cfg['input_channels'])]
    return labels

s = TRAIN_DS[0]
pts_in = s['input_geom'].cpu().numpy()
feat = s['x'].cpu().numpy()
pts_out = s['output_queries'].cpu().numpy()
y = s['y'].cpu().numpy()

labels = gino_input_channel_labels(CFG)
print('Input variant:', CFG['input_variant'])
print('Input channel labels:', labels)
print('Output channel labels:', ['Ux', 'Uy', 'Uz'])
print('Input geometry shape   [N_in,3]:', tuple(pts_in.shape))
print('Input feature shape    [N_in,C]:', tuple(feat.shape))
print('Output query shape     [N_out,3]:', tuple(pts_out.shape))
print('Output target shape    [N_out,3]:', tuple(y.shape))
print('Datapoints per sample -> N_in:', int(pts_in.shape[0]), 'N_out:', int(pts_out.shape[0]))

in_val = feat[:, 0]
out_speed = np.linalg.norm(y, axis=1)

max_pts = 7000
rng = np.random.default_rng(CFG['seed'])
idx_in = np.arange(len(pts_in))
if len(idx_in) > max_pts:
    idx_in = rng.choice(len(pts_in), size=max_pts, replace=False)
idx_out = np.arange(len(pts_out))
if len(idx_out) > max_pts:
    idx_out = rng.choice(len(pts_out), size=max_pts, replace=False)

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
sc1 = ax1.scatter(pts_in[idx_in, 0], pts_in[idx_in, 1], pts_in[idx_in, 2], c=in_val[idx_in], s=4, cmap='viridis', alpha=0.7)
ax1.set_title('Input particles/features in 3D')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
plt.colorbar(sc1, ax=ax1, fraction=0.04, pad=0.03)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
sc2 = ax2.scatter(pts_out[idx_out, 0], pts_out[idx_out, 1], pts_out[idx_out, 2], c=out_speed[idx_out], s=4, cmap='turbo', alpha=0.7)
ax2.set_title('Output query/velocity in 3D')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')
plt.colorbar(sc2, ax=ax2, fraction=0.04, pad=0.03)

fig.tight_layout()
save_fig(fig, 'data_profile_3d_pairs.png')
plt.show()


In [ ]:
# =========================
# IO Flow Demo: raw pair -> tensors -> GINO -> prediction
# =========================
print('Flow: raw npz -> RawPairDataset -> DataLoader batch -> GINO(input_geom, x, output_queries) -> prediction')

batch = next(iter(TRAIN_LOADER))
print('Batch input_geom shape  [B,N_in,3]:', tuple(batch['input_geom'].shape))
print('Batch feature x shape   [B,N_in,C]:', tuple(batch['x'].shape))
print('Batch output_queries    [B,N_out,3]:', tuple(batch['output_queries'].shape))
print('Batch target y shape    [B,N_out,3]:', tuple(batch['y'].shape))

with torch.no_grad():
    pred = predict(MODEL, batch).detach().cpu()

print('Model output shape      [B,N_out,3]:', tuple(pred.shape))
print('Input channels used:', CFG['input_channels'], 'variant:', CFG['input_variant'])
print('Output channels predicted:', ['Ux', 'Uy', 'Uz'])

pts = batch['output_queries'][0].cpu().numpy()
true_speed = torch.linalg.norm(batch['y'][0], dim=1).cpu().numpy()
pred_speed = torch.linalg.norm(pred[0], dim=1).numpy()
err = np.abs(pred_speed - true_speed)

max_pts = 7000
rng = np.random.default_rng(CFG['seed'])
idx = np.arange(len(pts))
if len(idx) > max_pts:
    idx = rng.choice(len(pts), size=max_pts, replace=False)

fig = plt.figure(figsize=(18, 5))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
sc1 = ax1.scatter(pts[idx, 0], pts[idx, 1], pts[idx, 2], c=true_speed[idx], s=4, cmap='turbo', alpha=0.7)
ax1.set_title('True speed on output queries')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
plt.colorbar(sc1, ax=ax1, fraction=0.04, pad=0.03)

ax2 = fig.add_subplot(1, 3, 2, projection='3d')
sc2 = ax2.scatter(pts[idx, 0], pts[idx, 1], pts[idx, 2], c=pred_speed[idx], s=4, cmap='turbo', alpha=0.7)
ax2.set_title('Pred speed on output queries')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')
plt.colorbar(sc2, ax=ax2, fraction=0.04, pad=0.03)

ax3 = fig.add_subplot(1, 3, 3, projection='3d')
sc3 = ax3.scatter(pts[idx, 0], pts[idx, 1], pts[idx, 2], c=err[idx], s=4, cmap='magma', alpha=0.7)
ax3.set_title('|Error| on output queries')
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')
plt.colorbar(sc3, ax=ax3, fraction=0.04, pad=0.03)

fig.tight_layout()
save_fig(fig, 'io_flow_demo.png')
plt.show()
